# 레슨 11 - 에이전트 간 (A2A) 프로토콜


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A 프로토콜이란?

이 **Agent-to-Agent (A2A) 프로토콜** 은 AI 에이전트가 통신하고,
서로를 발견하며, 협업할 수 있게 해주며 — 서로 다른 프레임워크로 구축되었거나 호스팅
된 다른 서비스에 있더라도.

핵심 개념:

- **발견** – 에이전트는 자신의 역량을 설명하는 *에이전트 카드*를 게시하여, 다른 에이전트(또는 오케스트레이터)가 작업에 적합한 전문가를 찾기
  쉽도록 합니다.
- **메시지 전달** – 에이전트들은 공통 프로토콜을 통해 구조화된 메시지를 교환하므로, 하나의
  에이전트로부터의 요청이 다른 에이전트에 의해 그 내부 구현과 상관없이
  이해되고 처리될 수 있습니다.
- **작업 수명주기** – A2A는 *제출됨*, *작업 중*, *완료됨*, 그리고
  *실패됨* 등의 상태를 정의하여, 오케스트레이터가 위임된 작업이 어떻게 진행되는지 완전히 파악할 수 있게 합니다.

이 수업에서는 A2A 스타일의 협업을 시뮬레이션하기 위해 세 명의 전문화된 여행 에이전트를 연결하여
각 에이전트가 자신의 전문 지식을 기여하고 결과를 다음 에이전트로 전달하는 워크플로에 통합합니다.


## 특화된 여행 에이전트 만들기


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 워크플로우를 통한 다중 에이전트 협업

우리는 세 에이전트를 A2A 메시지 전달을 반영하는 순차적 워크플로우로 연결합니다:

1. **CurrencyExchangeAgent**는 사용자 요청을 수신하고 환율 안내를 제공합니다.
2. **ActivityPlannerAgent**는 풍부해진 컨텍스트를 수신하고 활동 추천을 추가합니다.
3. **TravelManagerAgent**는 두 입력을 종합하여 최종 여행 요약을 작성합니다.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 프로덕션 환경에서 A2A 이해하기

In a production environment the A2A protocol unlocks powerful cross-service scenarios:

| Capability | Description |
|---|---|
| **프레임워크 간 상호운용성** | 하나의 프레임워크로 구축된 에이전트는 다른 어떤 A2A-compliant 프레임워크로 구축된 에이전트에게 작업을 위임할 수 있어, 진정한 조직 간 상호운용성을 가능하게 합니다. |
| **서비스 경계** | 에이전트는 별도의 마이크로서비스, 클라우드 리전 또는 심지어 다른 조직에 존재하면서도 원활하게 협업할 수 있습니다. |
| **동적 검색** | 오케스트레이터는 런타임에 Agent Card 레지스트리를 조회하여 특정 하위 작업에 가장 적합한 전문가를 찾을 수 있습니다. |
| **스트리밍 및 푸시 알림** | A2A는 실시간 진행 업데이트를 위해 Server-Sent Events (SSE)를 지원하며, 장시간 실행되는 작업에 대한 푸시 알림도 지원합니다. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
배포에서는 각 에이전트가 HTTP 엔드포인트를 노출하고, Agent Card를 발행하며, 통신
via the A2A JSON-RPC protocol.


## 요약

이 레슨에서 배운 내용:

1. **A2A 프로토콜이 무엇인지** — 에이전트 간 검색, 메시징,
   및 작업 관리.
2. **특화된 에이전트를 만드는 방법** — 통화 환전 에이전트, 활동 계획 에이전트,
   및 여행 관리자 오케스트레이터.
3. **워크플로에 에이전트를 연결하는 방법** — `WorkflowBuilder`를 사용하여 순차적인
   에이전트 간 메시지 전달을 모델링합니다.
4. **프로덕션 환경에서 A2A가 작동하는 방식** — 프레임워크 간, 서비스 간 협업을 가능하게 함
   동적 검색 및 스트리밍 업데이트로.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
면책사항:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있지만, 자동 번역에는 오류나 부정확한 내용이 포함될 수 있음을 알려드립니다. 원문(원어) 문서가 권위 있는 출처로 간주되어야 합니다. 중요한 정보의 경우 전문 번역가의 번역을 권장합니다. 이 번역의 사용으로 인해 발생하는 어떠한 오해나 잘못된 해석에 대해서도 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
